# ROUGE Evaluation: VLM-Generated vs Human Insights

Pipeline:
1. Fetch ground truth (`insight_part_1`) and image from Supabase
2. Run Qwen2.5-VL-3B-Instruct on the dashboard screenshot
3. Evaluate AI-generated insight against ground truth using ROUGE

In [ ]:
!pip install -q evaluate rouge_score transformers qwen-vl-utils accelerate

In [ ]:
import io
import requests
import pandas as pd
from PIL import Image
from google.colab import userdata

SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")
BUCKET_NAME = "superstore"

HEADERS = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
}

print(f"URL: {SUPABASE_URL}")
print(f"Key loaded: {SUPABASE_KEY is not None}")
print("Environment loaded.")

In [ ]:
METADATA_ID = "8040a121-e403-4381-bcfd-8d32fb05c5b4"
BUCKET_PATH = f"screenshots/{METADATA_ID}.png"

# fetch ground truth via REST API
resp = requests.get(
    f"{SUPABASE_URL}/rest/v1/human_insights",
    headers=HEADERS,
    params={
        "select": "insight_part_1",
        "metadata_id": f"eq.{METADATA_ID}",
    },
)
resp.raise_for_status()

ground_truth = resp.json()[0]["insight_part_1"]
print(f"Ground truth ({len(ground_truth)} chars):")
print(ground_truth)

In [ ]:
# download image from supabase storage via REST
img_resp = requests.get(
    f"{SUPABASE_URL}/storage/v1/object/{BUCKET_NAME}/{BUCKET_PATH}",
    headers=HEADERS,
)
img_resp.raise_for_status()

image = Image.open(io.BytesIO(img_resp.content)).convert("RGB")
print(f"Image size: {image.size}")
image

In [ ]:
from transformers import pipeline
from huggingface_hub import login
from google.colab import userdata
import torch

login(token=userdata.get("HF_TOKEN"))

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

pipe = pipeline(
    "image-text-to-text",
    model=MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Pipeline loaded: {MODEL_ID}")

In [ ]:
PROMPT = """You are an analyst describing an SALES OVERVIEW dashboard.
Analyze the dashboard chart-by-chart in fixed Z-pattern order (left→right, top→bottom). Treat the top scoreboard if any (e.g. sales, profit, returns, quantity, customers) as one chart.
For each chart, write a tight 3-4 sentence paragraph that:
\t1.\tStates the key quantitative facts (numbers, comparisons to targets, highest/lowest values)
\t2.\tDescribes the main trends, patterns, or relationships visible
\t3.\tExplains business implications or recommended actions
Format exactly like this:
Chart 1 – Scoreboard Overview
Sales reached £733K against target, with profit at £93K, returns 4.7K units, and quantity 12.5K. The business shows growth across sales and quantity yoy, though returns are also rising. This suggests strong demand but potential quality or service issues that could impact margins if unaddressed.
  
Chart 2 – [Your chart description]
[3-4 sentences following the same 1-2-3 flow]

Rules:
- Cover EVERY chart in Z-order; do not skip any
- Max 3 sentences per chart
- No Level 1 content (axes, colors, chart types)
- No external business knowledge—base everything on visible data
- Use crisp, executive-style language
- TOTAL OUTPUT MUST NOT EXCEED 2000 CHARACTERS (including spaces)"""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT},
        ],
    }
]

torch.manual_seed(42)
pipe_out = pipe(
    messages,
    max_new_tokens=1024,
    clean_up_tokenization_spaces=True,
    do_sample=False,
    num_beams=5,
    repetition_penalty=2.0,
)

ai_insight = pipe_out[0]["generated_text"][-1]["content"]

print(f"AI-generated insight ({len(ai_insight)} chars):")
print(ai_insight)

In [ ]:
import evaluate

# ROUGE evaluation
rouge_metric = evaluate.load("rouge")
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

scores = rouge_metric.compute(
    predictions=[ai_insight],
    references=[ground_truth],
)

results_df = pd.DataFrame(
    {"metric": rouge_names, "score": [scores[k] for k in rouge_names]}
)
results_df

In [ ]:
# side-by-side comparison
comparison_df = pd.DataFrame({
    "source": ["Ground Truth", "AI-Generated"],
    "insight": [ground_truth, ai_insight],
    "char_count": [len(ground_truth), len(ai_insight)],
})
comparison_df